# Open Challenge 2 — Transpilation Battle
**Mission:** optimize fixed quantum circuits for a **5-qubit linear** simulated device. **Estimated time:** 5–10 hours. **100 points.**

**Constraints:** physical edges 0–1–2–3–4; bidirectional `cx`; basis `rz`, `sx`, `x`, `cx`. No cloud, API key, or real hardware. You may use built-in Qiskit passes or custom circuit rewrites, but explain your method.

**Deliverables:** optimization code, metrics CSV/table, before/after circuits, correctness checks, README, five-minute demo. Organizers should use additional hidden circuits to test generality.

**Important:** compare *unitary* circuits before measurements. Account for logical-to-physical layouts when testing equivalence. A better gate count with incorrect behavior earns zero optimization points for that case.

## Local setup
Run once with internet, then work fully offline. No IBM account or token. Python 3.10+ recommended.

```python
%pip install qiskit qiskit-aer numpy scipy matplotlib networkx
```
Restart kernel after installation. The following cells assume packages are installed.

In [ ]:
%pip install qiskit qiskit-aer numpy scipy matplotlib networkx

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, Operator, state_fidelity
from qiskit_aer import AerSimulator
SEED = 42
SHOTS = 2048
print("Ready: local Qiskit Aer; seed", SEED)

In [ ]:
from qiskit.transpiler import CouplingMap
from qiskit.circuit.library import QFT
from qiskit.quantum_info import Operator
from time import perf_counter

EDGES = [(i, i+1) for i in range(4)] + [(i+1, i) for i in range(4)]
COUPLING = CouplingMap(EDGES)
BASIS = ["rz", "sx", "x", "cx"]

def benchmark_circuits():
    cases = {}
    a = QuantumCircuit(5)
    a.h(0); a.cx(0,4); a.cx(0,4); a.h(0)
    a.cx(1,3); a.cx(3,4)
    cases["redundant_and_remote"] = a
    b = QuantumCircuit(5)
    b.h(0)
    for i in range(1,5): b.cx(0,i)
    cases["ghz_star"] = b
    c = QuantumCircuit(5)
    c.h(0); c.cx(0,4); c.h(2); c.cx(2,4); c.cz(1,3)
    cases["remote_entanglers"] = c
    d = QuantumCircuit(5)
    for i in range(5): d.h(i)
    for i in [0,1,2,3]: d.cx(i,i+1)
    for i in [3,2,1,0]: d.cx(i,i+1)
    cases["cancel_chain"] = d
    return cases

CASES = benchmark_circuits()
print([(k, c.num_qubits) for k,c in CASES.items()])

## Phase 1 — Baselines
Transpile each case at optimization levels 0–3 with the same seed. Measure CX count, depth, and wall-clock compile time. Qiskit may optimize away entire subcircuits; that is expected.

In [ ]:
def metrics(circuit):
    return {"cx": circuit.count_ops().get("cx", 0), "depth": circuit.depth(), "size": circuit.size()}

baselines = {}
for name, circuit in CASES.items():
    for level in range(4):
        t0 = perf_counter()
        out = transpile(circuit, basis_gates=BASIS, coupling_map=COUPLING,
                        optimization_level=level, seed_transpiler=SEED)
        baselines[(name,level)] = out
        print(name, "level", level, metrics(out), "seconds", round(perf_counter()-t0,3))

## Phase 2 — Your optimizer
Implement `optimize_for_device(circuit)`. You may choose initial layout, routing, optimization passes, or circuit identities. The returned circuit must use the target basis and legal physical edges. For reproducibility, fix seeds.

**Suggested ideas:** cancel inverse gates, commute gates to expose cancellations, change initial layout, compare routing methods, synthesize Clifford blocks. Don't optimize against only the provided circuits.

In [ ]:
def optimize_for_device(circuit):
    # TODO: replace this baseline with your strategy
    return transpile(circuit, basis_gates=BASIS, coupling_map=COUPLING,
                     optimization_level=3, seed_transpiler=SEED)

optimized = {name: optimize_for_device(c) for name,c in CASES.items()}

## Phase 3 — Verify correctness and physical legality
A routed circuit can have a nontrivial initial/final layout. For the provided small benchmarks, use `Operator.from_circuit(..., ignore_set_layout=True)`? **No**: ignoring layouts can silently hide permutation issues. Instead, use `Operator.from_circuit(circuit)` with layout metadata where supported, and verify on test inputs as well. Qiskit version behavior differs; report how you handled layout. The supplied helper uses `Operator.from_circuit` for the original and `Operator.from_circuit` for the transpiled circuit, which incorporates layout metadata in supported versions.

Do not compare only `|00000⟩` measurements: that can miss phase and unitary errors.

In [ ]:
def legal_physical_gates(c):
    edges = set(EDGES)
    for inst in c.data:
        op = inst.operation
        if op.name not in BASIS and op.name not in {"barrier"}:
            return False, f"Unsupported operation: {op.name}"
        if op.name == "cx":
            pair = tuple(c.find_bit(q).index for q in inst.qubits)
            if pair not in edges:
                return False, f"Illegal CX edge: {pair}"
    return True, "OK"

def equivalent(original, candidate):
    # Qiskit Operator.from_circuit handles circuit layout metadata.
    try:
        return Operator.from_circuit(original).equiv(Operator.from_circuit(candidate))
    except Exception as exc:
        print("Equivalence check needs layout inspection:", exc)
        return False

for name, candidate in optimized.items():
    legal, message = legal_physical_gates(candidate)
    same = equivalent(CASES[name], candidate)
    print(name, "equivalent:", same, "legal:", legal, message,
          "baseline:", metrics(baselines[(name,3)]), "team:", metrics(candidate))

## Phase 4 — Compare and report
Calculate improvements relative to optimization level 3. Avoid division by zero if CX count or depth is zero. Report both metrics, even if one worsens. Organizers can score improvement using normalized ratios, but **equivalence and connectivity are hard gates**.

**Advanced:** inject per-edge two-qubit error rates and compare predicted or simulated output quality; write a custom `TransformationPass` and benchmark it on unseen circuits.

In [ ]:
for name, c in optimized.items():
    b = metrics(baselines[(name,3)])
    t = metrics(c)
    print(f"{name}: CX {b['cx']} -> {t['cx']}, depth {b['depth']} -> {t['depth']}")
# TODO: save results to CSV and include optimization methodology.

## Judging (100)
| Criterion | Points |
|---|---:|
| Equivalence and hardware validity | 25 |
| Two-qubit gate reduction | 25 |
| Depth reduction | 20 |
| Quality and generality of method | 15 |
| Benchmarks and reproducibility | 10 |
| Explanation | 5 |

**Submission:** `optimize_for_device` function, report with all four benchmark cases, proof/check of equivalence, package versions, README. Hidden benchmarks may be used for final judging.